In [1]:
%load_ext autoreload
%autoreload 2

[PyTorch Tutorial](https://docs.pytorch.org/tutorials/intermediate/reinforcement_q_learning.html)

# Imports

In [2]:
from src.app import run
from src.game import Screen, Game
from src.config.enums import Direction
from src.ai.memory import ReplayMemory, Transition

C:\Users\alast\dev\uni-dev\mat502-ai\mat501-coursework-project\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.3)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
import math
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count
import numpy as np
from tqdm import tqdm

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Config

## Matplotlib

In [5]:
is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display

## Pytorch

In [6]:
x = torch.rand(5, 3)
print(x)

tensor([[0.3240, 0.9418, 0.2303],
        [0.9772, 0.8742, 0.4085],
        [0.8000, 0.0571, 0.5265],
        [0.9288, 0.4346, 0.1404],
        [0.3720, 0.4614, 0.5969]])


In [7]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)
print(device)

cuda


# Experiment

## Model

In [8]:
# Conv from https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html
class DQN(nn.Module):
    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(n_observations, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, 128)
        self.layer4 = nn.Linear(128, n_actions)
    

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        x = F.relu(self.layer3(x))
        return self.layer4(x)

## Hyperparameters

In [9]:
BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.01
EPS_DECAY = 2500
TAU = 0.005
LR = 3e-4
MAX_STEPS = 500
GRADIENT_CLIP = 100

## Config

### Game

In [10]:
screen = Screen(is_draw_enabled=False)
game = Game(
    screen,
    "MAT501 - Coursework",
    is_draw_enabled=False,
    framerate=100000,
)

### Utility Functions

In [11]:
def select_action(state, _steps_done):
    sample = random.random()
    eps_threshold = EPS_END + (EPS_START - EPS_END) * \
        math.exp(-1. * _steps_done / EPS_DECAY)
    if sample > eps_threshold:
        with torch.no_grad():
            return policy_net(state).max(1).indices.view(1, 1)
    else:
        return torch.tensor([[random.randint(0, len(Direction)-1)]], device=device, dtype=torch.long)

In [12]:
def plot_durations(_episode_durations, show_result=False):
    plt.figure(1)
    # durations_t = torch.tensor(max_snake_lengths, dtype=torch.float)
    durations_t = torch.tensor(_episode_durations, dtype=torch.float)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel('Duration')
    plt.plot(durations_t.numpy())
    if len(durations_t) >= 100:
        means = durations_t.unfold(0, 100, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())

    plt.pause(0.001)
    if is_ipython:
        if not show_result:
            display.display(plt.gcf())
            display.clear_output(wait=True)
        else:
            display.display(plt.gcf())

## Training

In [13]:
def optimize_model(_memory):
    if len(_memory) < BATCH_SIZE:
        return
    transitions = _memory.sample(BATCH_SIZE)
    batch = Transition(*zip(*transitions))
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                          batch.next_state)), device=device, dtype=torch.bool)
    non_final_next_states = torch.cat([s for s in batch.next_state
                                                if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)
    state_action_values = policy_net(state_batch).gather(1, action_batch)
    next_state_values = torch.zeros(BATCH_SIZE, device=device)
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), GRADIENT_CLIP)
    optimizer.step()

In [14]:
n_actions = len(Direction)
state = game.training_ai_get_current_snake_state()
n_observations = len(state)
n_states = max(state) + 1

policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
memory = ReplayMemory(10000)

In [15]:
steps_done = 0
max_snake_lengths = []
episode_durations = []

In [16]:
if torch.cuda.is_available() or torch.backends.mps.is_available():
    num_episodes = 200
else:
    num_episodes = 50

for i_episode in range(num_episodes):
    screen = Screen(is_draw_enabled=False)
    game = Game(
        screen,
        "MAT501 - Coursework",
        is_draw_enabled=False,
        framerate=10000000,
    )
    game_over = False
    
    for t in count():
        if t > MAX_STEPS:
            break

        game.training_ai_frame()
        game.training_ai_step(Direction.UP)
        state = game.training_ai_get_current_snake_state()
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

        action = select_action(state, steps_done)
        steps_done += 1
        observation, reward, game_over = game.training_ai_step(list(Direction)[action.item()])
        reward = torch.tensor([reward], device=device)

        if game_over:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        memory.push(state, action, next_state, reward)
        state = next_state
        optimize_model(memory)

        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key]*TAU + target_net_state_dict[key]*(1-TAU)
        target_net.load_state_dict(target_net_state_dict)

        if game_over:
            max_snake_lengths.append(max(len(game.snakes[0].body), len(game.snakes[1].body)))
            episode_durations.append(t+1)
            plot_durations(episode_durations)
            game.training_ai_end_frame()
            break

        game.training_ai_end_frame()

plot_durations(episode_durations, show_result=True)
plt.ioff()
plt.show()
torch.save(policy_net.state_dict(), "models/base_policy_net.pth")

KeyboardInterrupt: 

<Figure size 640x480 with 0 Axes>

In [ ]:
len(episode_durations)

In [ ]:
MAX_STEPS

In [ ]:
i_episode

In [21]:
memory.sample(1)

[Transition(state=tensor([[0.7000, 0.8000, 0.7500, 0.7500]], device='cuda:0'), action=tensor([[2]], device='cuda:0'), next_state=tensor([[0.6500, 0.8000, 0.7500, 0.7500]], device='cuda:0'), reward=tensor([-0.2500], device='cuda:0'))]